# 第 8 周：Shine Your Eye — 尼日利亚多代理（产品就绪）

## 练习目标

搭建一个**面向产品**的多代理助手，覆盖奈拉汇率、本地优惠与常见问题（NIN、拉各斯等）：

- **Router Agent** — 意图分类：`naira_rate` | `deals` | `faq` | `general`
- **Naira Agent** — 奈拉/美元汇率 + CBN 风格说明（Pydantic 结构化）
- **DealScanner Agent** — 尼日利亚风优惠（Jumia/Konga）；结构化输出
- **RAG Agent** — ChromaDB 常见问题；从向量库取答案
- **Orchestrator** — 跑路由器、分派代理、返回格式化流程与答案
- **Gradio UI** — 用英语或 Pidgin 提问；查看代理流程 + 答案

## 和本课 Week 8 的关系

类型化代理、Pydantic schema、日志、RAG（ChromaDB）、可选 Telegram 推送；无服务器场景可把代理部署到 Modal 再由协调器调用。

## 怎么跑

1. 按顺序运行单元格  
2. 在 `.env` 设置 `OPENROUTER_API_KEY`  
3. **Telegram（可选）：** 经 [BotFather](https://t.me/BotFather) 建机器人，在 `.env` 设 `TELEGRAM_BOT_TOKEN` 与 `TELEGRAM_CHAT_ID`；UI 勾选「Send to Telegram」即可推送  


In [ ]:
# ========== 导入依赖 + 从 .env 读取密钥与模型配置 ==========

# os：读环境变量
import os
# json：解析路由器返回的意图 JSON
import json
# logging：彩色代理日志的底层
import logging
# requests：调用 Telegram Bot API
import requests
# numpy：嵌入矩阵 → t-SNE/PCA
import numpy as np
# 类型注解：可选参数与列表
from typing import Optional, List
# Pydantic：结构化意图 / 汇率 / 优惠
from pydantic import BaseModel, Field
# OpenAI 兼容客户端（此处指向 OpenRouter）
from openai import OpenAI
# 加载 .env，避免密钥进仓库
from dotenv import load_dotenv
# ChromaDB：FAQ 向量库
import chromadb
from chromadb.config import Settings
# Gradio：聊天 + 可视化 UI
import gradio as gr
# 降维：样本少用 PCA，否则 t-SNE
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
# Plotly：3D 散点图给 Gradio Plot
import plotly.graph_objects as go

# 读取 .env 到进程环境
load_dotenv()
# OpenRouter API Key（主路径必需）
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
# Telegram 可选推送
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")
# OpenRouter Chat Completions 基址（URL 保持原样）
BASE_URL = "https://openrouter.ai/api/v1"
# 路由/对话使用的模型 id（保持原样）
MODEL = "openai/gpt-4o-mini"


In [ ]:
# ========== 代理基类：统一名字、ANSI 颜色与日志 ==========

# 配置根日志格式：时间戳 + 消息
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s")
# 本模块 logger（基类里实际用 logging.info）
logger = logging.getLogger(__name__)

class Agent:
    # 子类覆盖：代理显示名
    name = ""
    # 默认前景色（白）
    color = "\033[37m"
    # 常用 ANSI 颜色常量，供子类选用
    RED, GREEN, YELLOW, BLUE, CYAN, RESET = "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[36m", "\033[0m"

    def log(self, msg: str):
        # 带颜色前缀打印，便于在终端区分哪个代理在说话
        logging.info(f"{self.color}[{self.name}] {msg}{self.RESET}")


In [ ]:
# ========== Pydantic 数据模型：意图 / 汇率 / 优惠列表 ==========

class RouterIntent(BaseModel):
    # 四选一意图标签（description 给 schema/文档用，勿改英文）
    intent: str = Field(description="One of: naira_rate, deals, faq, general")
    # 简短分类理由
    reason: str = Field(description="Brief reason for this intent")

class NairaRate(BaseModel):
    # 1 USD 兑多少 NGN
    rate_usd_ngn: float = Field(description="Naira per 1 USD")
    # 来源说明，如 CBN / 平行市场
    source: str = Field(description="e.g. CBN, parallel market")
    # 补充短注
    note: str = Field(description="Short note")

class NigerianDeal(BaseModel):
    # 商品标题
    title: str = Field(description="Product title")
    # 奈拉标价
    price_ngn: float = Field(description="Price in Naira")
    # 品类
    category: str = Field(description="e.g. Phones, Fashion")
    # 商品链接
    url: str = Field(description="Link to deal")

class DealList(BaseModel):
    # 最多约 5 条优惠
    deals: List[NigerianDeal] = Field(description="Up to 5 Nigerian-style deals")


In [ ]:
# ========== LLM 工具：可选 JSON Schema 约束的 Chat Completions ==========

def llm_json(client: OpenAI, system: str, user: str, response_format: Optional[dict] = None):
    # 基础请求：模型 + system/user 两条消息
    kwargs = {"model": MODEL, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    # 若传入 schema，则挂上 OpenAI 风格 json_schema 响应格式
    if response_format:
        kwargs["response_format"] = {"type": "json_schema", "json_schema": {"name": "r", "strict": True, "schema": response_format}}
    # 发起补全
    r = client.chat.completions.create(**kwargs)
    # 返回助手文本（通常是 JSON 字符串）
    return r.choices[0].message.content


In [ ]:
# ========== RouterAgent：把用户问句分类成意图 ==========

class RouterAgent(Agent):
    name = "Router"
    color = Agent.GREEN

    def route(self, query: str, client: OpenAI) -> RouterIntent:
        # 日志只截前 60 字符，避免刷屏
        self.log(f"Routing: {query[:60]}...")
        # 系统提示要求 JSON 意图（prompt 英文保留）
        system = "You classify user intent. Reply with JSON: {intent: one of naira_rate, deals, faq, general, reason: short reason}."
        user = f"User query: {query}"
        # 调用通用 llm_json（此处未传 schema，靠提示约束）
        raw = llm_json(client, system, user)
        try:
            # 解析 JSON → RouterIntent
            d = json.loads(raw)
            out = RouterIntent(intent=d.get("intent", "general"), reason=d.get("reason", ""))
        except Exception:
            # 解析失败时退回 general，便于流水线继续
            out = RouterIntent(intent="general", reason="parse failed")
        self.log(f"Intent: {out.intent}")
        return out


In [ ]:
# ========== NairaAgent：模拟汇率查询并格式化回答 ==========

class NairaAgent(Agent):
    name = "Naira"
    color = Agent.CYAN

    def get_rate(self, client: OpenAI) -> NairaRate:
        # 教学演示：固定模拟汇率，不打真实 API
        self.log("Fetching Naira/USD rate (simulated).")
        return NairaRate(rate_usd_ngn=1580.0, source="CBN (simulated)", note="Official window. Parallel market often higher.")

    def run(self, query: str, client: OpenAI) -> str:
        # 取结构化汇率，再拼成可读句子
        rate = self.get_rate(client)
        return f"1 USD = ₦{rate.rate_usd_ngn:,.2f} ({rate.source}). {rate.note}"


In [ ]:
# ========== DealScannerAgent：返回模拟的本地优惠列表 ==========

class DealScannerAgent(Agent):
    name = "DealScanner"
    color = Agent.YELLOW

    # 演示用 mock 优惠（标题/价格/链接保持原样）
    MOCK_DEALS = [
        NigerianDeal(title="Samsung A54 5G", price_ngn=285000, category="Phones", url="https://jumia.com.ng/example1"),
        NigerianDeal(title="Nivea Body Lotion 400ml", price_ngn=4500, category="Beauty", url="https://konga.com/example2"),
        NigerianDeal(title="Nigerian Jollof Rice Spice Pack", price_ngn=2500, category="Food", url="https://jumia.com.ng/example3"),
    ]

    def run(self, query: str, client: OpenAI) -> str:
        self.log("Scanning Nigerian deals (mock).")
        # 每条一行 bullet；千分位格式化奈拉价格
        lines = [f"• {d.title} – ₦{d.price_ngn:,.0f} ({d.category})" for d in self.MOCK_DEALS]
        return "Deals today:\n" + "\n".join(lines)


In [ ]:
# ========== 构建 ChromaDB FAQ 集合（空库时灌入种子文档）==========

# 四条英文 FAQ 原文（作为 RAG 文档内容，勿翻译改写）
NIGERIAN_FAQS = [
    "NIN is National Identification Number. You can enroll at NIMC offices or approved centers. Bring BVN if you have it.",
    "Lagos has 20 LGAs. Popular areas: Ikeja, Lekki, Victoria Island, Surulere, Yaba, Ajah.",
    "CBN sets official Naira rate. Parallel (black) market rate is often higher. Check CBN website for official.",
    "BVN links your bank accounts. Get it at any bank. You need valid ID and biometrics.",
]

def build_rag():
    # 关闭匿名遥测的内存/本地客户端
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    # 取得或创建名为 naija_faq 的集合
    coll = client.get_or_create_collection("naija_faq", metadata={"description": "Nigerian FAQs"})
    # 仅首次灌入，避免重复 add
    if coll.count() == 0:
        for i, text in enumerate(NIGERIAN_FAQS):
            coll.add(ids=[str(i)], documents=[text], metadatas=[{"id": i}])
    return coll

# 模块加载时建好全局 collection，供 RAGAgent / 可视化使用
rag_collection = build_rag()


In [ ]:
# ========== RAGAgent：向量检索 FAQ 并直接返回文档片段 ==========

class RAGAgent(Agent):
    name = "RAG"
    color = Agent.BLUE

    def __init__(self, collection):
        # 持有 Chroma collection 引用
        self.coll = collection

    def run(self, query: str, client: OpenAI) -> str:
        self.log("Querying Nigerian FAQ RAG.")
        # 按文本查询，取 top-2
        res = self.coll.query(query_texts=[query], n_results=2)
        docs = res["documents"][0] if res["documents"] else []
        if not docs:
            # 无命中时的英文提示（行为相关文案保留）
            return "No matching FAQ. Try asking about NIN, Lagos, CBN, or BVN."
        # 多条文档用换行拼成答案
        return "\n".join(docs)


In [ ]:
# ========== RAG 嵌入的 3D 可视化（t-SNE / PCA → Plotly）==========

# 与种子 FAQ 顺序对应的标签
FAQ_LABELS = ["NIN", "Lagos", "CBN", "BVN"]
# 每个点的颜色
COLORS = ["#9b59b6", "#3498db", "#e67e22", "#2ecc71"]

def get_rag_viz_plot(collection):
    """Build 3D scatter of FAQ embeddings (t-SNE/PCA). Returns Plotly figure for Gradio."""
    # 取出嵌入、原文与元数据
    data = collection.get(include=["embeddings", "documents", "metadatas"])
    if not data["ids"]:
        # 空库：占位图
        fig = go.Figure()
        fig.add_annotation(text="No RAG data yet.", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(title="RAG Embedding Space (empty)")
        return fig
    # 转成 float64 矩阵
    emb = np.array(data["embeddings"], dtype=np.float64)
    n = len(emb)
    if n <= 3:
        # 点太少时 t-SNE 不稳，改用 PCA
        coords = PCA(n_components=min(3, emb.shape[1])).fit_transform(emb)
    else:
        # perplexity 必须 < n
        coords = TSNE(n_components=3, random_state=42, perplexity=min(5, n - 1)).fit_transform(emb)
    colors = [COLORS[i % len(COLORS)] for i in range(n)]
    labels = [FAQ_LABELS[i] if i < len(FAQ_LABELS) else f"Doc {i}" for i in range(n)]
    fig = go.Figure(data=[
        go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
            mode="markers+text",
            marker=dict(size=12, color=colors, opacity=0.9),
            text=labels, textposition="top center",
        )
    ])
    fig.update_layout(
        title="RAG FAQ Embedding Space (t-SNE 3D)",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", bgcolor="#1e1e2e"),
        paper_bgcolor="#1e1e2e", font=dict(color="#eee"), height=450,
    )
    return fig


In [ ]:
# ========== Orchestrator：路由 → 分派 → 拼装流程 Markdown ==========

class Orchestrator:
    def __init__(self):
        # 有 Key 才建 OpenRouter 客户端；否则后续提示配置
        self.client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL) if OPENROUTER_API_KEY else None
        self.router = RouterAgent()
        self.naira = NairaAgent()
        self.deals = DealScannerAgent()
        self.rag = RAGAgent(rag_collection)

    def run(self, query: str) -> tuple:
        """Returns (flow_md, answer) for Gradio visualization."""
        steps = []
        if not self.client:
            return "**Flow:** (no API key)\n", "Set OPENROUTER_API_KEY in .env to use agents."
        # 记录输入摘要
        steps.append(f"📥 **Input:** {query[:80]}...")
        # 路由器决定 intent
        intent = self.router.route(query, self.client)
        steps.append(f"🟢 **Router** → intent: `{intent.intent}` ({intent.reason})")
        if intent.intent == "naira_rate":
            steps.append("🔵 **Naira** → fetching rate...")
            out = self.naira.run(query, self.client)
            steps.append("🔵 **Naira** → done")
        elif intent.intent == "deals":
            steps.append("🟡 **DealScanner** → scanning...")
            out = self.deals.run(query, self.client)
            steps.append("🟡 **DealScanner** → done")
        elif intent.intent == "faq":
            steps.append("🔷 **RAG** → querying FAQ...")
            out = self.rag.run(query, self.client)
            steps.append("🔷 **RAG** → done")
        else:
            # general 或其它：给引导文案
            out = f"Intent '{intent.intent}' – try: naira rate, deals, or NIN/Lagos/CBN/BVN."
        flow_md = "**Agent flow**\n\n" + "\n".join(steps)
        return flow_md, out


In [ ]:
# 快速测试（无 Gradio）：可取消注释，直接跑 Orchestrator 一次
# print(chat("美元兑奈拉多少钱？"))
# print(chat("我们是 NIN 吗？"))


In [ ]:
# ========== 可选：把问答摘要推送到 Telegram ==========

def send_report_to_telegram(query: str, flow_md: str, answer: str) -> str:
    """Send a short report to Telegram. Returns status message."""
    # 未配置则直接返回提示，不抛异常
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return "Telegram not configured (set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in .env)."
    # Markdown 风格短报告（截断防超长）
    text = (
        f"🇳🇬 *Shine Your Eye – Report*\n\n"
        f"*Query:* {query[:200]}\n\n"
        f"*Answer:*\n{answer[:3500]}"
    )
    # Bot API sendMessage 端点（URL 模式保持原样）
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        r = requests.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": text, "parse_mode": "Markdown"}, timeout=10)
        if r.ok:
            return "✅ Report sent to Telegram."
        return f"❌ Telegram error: {r.status_code} {r.text[:100]}"
    except Exception as e:
        return f"❌ Telegram error: {e}"


In [ ]:
# ========== Gradio UI：聊天 Tab + RAG 嵌入可视化 Tab ==========

# 懒加载单例，避免重复建 Orchestrator
_orch = None
def get_orch():
    global _orch
    if _orch is None:
        _orch = Orchestrator()
    return _orch

def chat(query: str, send_to_telegram: bool):
    # 跑多代理流水线
    flow_md, answer = get_orch().run(query)
    telegram_status = ""
    if send_to_telegram:
        # 勾选时才推送
        telegram_status = send_report_to_telegram(query, flow_md, answer)
    return flow_md, answer, telegram_status

with gr.Blocks(title="Shine Your Eye", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🇳🇬 Shine Your Eye – Nigerian Multi-Agent")
    with gr.Tabs():
        with gr.TabItem("Chat"):
            gr.Markdown("Ask in English or Pidgin. **Agent flow** shows which agents ran.")
            with gr.Row():
                inp = gr.Textbox(placeholder="e.g. How much is dollar to naira?", label="Ask", scale=2)
                send_tg = gr.Checkbox(label="📱 Send to Telegram", value=False)
                btn = gr.Button("Go", variant="primary")
            with gr.Row():
                flow_out = gr.Markdown(label="Agent flow", value="*Agent steps will appear here.*")
                ans_out = gr.Textbox(label="Answer", lines=8, interactive=False)
            tg_status = gr.Textbox(label="Telegram", value="", interactive=False)
            def run_chat(q, stg):
                # 薄包装，方便绑到 click/submit
                flow_md, answer, status = chat(q, stg)
                return flow_md, answer, status
            btn.click(fn=run_chat, inputs=[inp, send_tg], outputs=[flow_out, ans_out, tg_status])
            inp.submit(fn=run_chat, inputs=[inp, send_tg], outputs=[flow_out, ans_out, tg_status])
        with gr.TabItem("RAG embedding viz"):
            gr.Markdown("3D view of FAQ embeddings (t-SNE). Each point = one RAG document.")
            viz_btn = gr.Button("Show 3D embedding plot", variant="secondary")
            viz_plot = gr.Plot(label="RAG vector space")
            # lambda 闭包当前 rag_collection
            viz_btn.click(fn=lambda: get_rag_viz_plot(rag_collection), inputs=None, outputs=viz_plot)
# 无头环境不自动开浏览器
app.launch(inbrowser=False)
